# CitiBike (NYC)

In [ ]:
import pandas as pd
import celldega as dega

In [ ]:
from glob import glob

paths = sorted(glob('data/bikes/citibikes/202603-citibike-tripdata/202603-citibike-tripdata_*.csv'))
df = pd.concat((pd.read_csv(p) for p in paths), ignore_index=True)

print(f'Loaded {len(paths)} files')
paths

In [ ]:
df.shape[0]/1e6

In [ ]:
df.head()

In [ ]:
# Build a canonical station table from both start and end station fields
start_stations = (
    df[["start_station_id", "start_station_name", "start_lat", "start_lng"]]
    .rename(
        columns={
            "start_station_id": "station_id",
            "start_station_name": "station_name",
            "start_lat": "lat",
            "start_lng": "lng",
        }
    )
)

end_stations = (
    df[["end_station_id", "end_station_name", "end_lat", "end_lng"]]
    .rename(
        columns={
            "end_station_id": "station_id",
            "end_station_name": "station_name",
            "end_lat": "lat",
            "end_lng": "lng",
        }
    )
)

stations = (
    pd.concat([start_stations, end_stations], ignore_index=True)
    .dropna(subset=["station_id", "station_name", "lat", "lng"])
    .groupby(["station_id", "station_name"], as_index=False)[["lat", "lng"]]
    .mean()  # average in case of minor coordinate jitter
    .sort_values("station_name")
    .reset_index(drop=True)
)

stations.head(), stations.shape

# Transition counts: rows = destination station, columns = origin station
valid_trips = df.dropna(
    subset=["start_station_id", "start_station_name", "end_station_id", "end_station_name"]
).copy()

valid_trips["origin"] = valid_trips["start_station_name"].astype(str)
valid_trips["destination"] = valid_trips["end_station_name"].astype(str)

transition_counts = pd.crosstab(valid_trips["destination"], valid_trips["origin"])
transition_prob = transition_counts.div(transition_counts.sum(axis=0), axis=1).fillna(0)

print("Stations table shape:", stations.shape)
print("Transition matrix shape:", transition_prob.shape)
print("Column sums (first 10):")
print(transition_prob.sum(axis=0).head(10))

In [ ]:
# Optional: build metadata aligned to transition_prob labels for downstream clustering
station_lookup = stations.copy()
station_lookup["label"] = station_lookup["station_name"].astype(str)
# Same name can appear with different station_id; crosstab merges those into one label — collapse here so index is unique
station_lookup = (
    station_lookup.groupby("label", as_index=False)
    .agg(lat=("lat", "mean"), lng=("lng", "mean"), station_id=("station_id", "first"))
    .set_index("label")
)

origin_station_meta = station_lookup.reindex(transition_prob.columns)
destination_station_meta = station_lookup.reindex(transition_prob.index)

origin_station_meta.head(), destination_station_meta.head()

In [ ]:
stations

In [ ]:
transition_prob

In [ ]:
stations.plot(kind='scatter', x='lng', y='lat')

In [ ]:
def _station_clusters_from_matrix_dendrogram(mat, n_clusters: int = 20) -> dict[str, int]:
    """Flat clusters by cutting the same hierarchical linkage as the Clustergram (col dendrogram, then row)."""
    from scipy.cluster.hierarchy import fcluster

    import numpy as np

    if mat is None or getattr(mat, "data", None) is None:
        return {}
    linkage_bundle = mat.viz.get("linkage") or {}
    for axis in ("col", "row"):
        Z = np.asarray(linkage_bundle.get(axis) or [], dtype=float)
        if Z.size == 0:
            continue
        try:
            labels = fcluster(Z, t=n_clusters, criterion="maxclust")
        except Exception:
            continue
        names = (
            mat.data.columns.astype(str).tolist()
            if axis == "col"
            else mat.data.index.astype(str).tolist()
        )
        if len(labels) != len(names):
            continue
        return {str(a): int(b) for a, b in zip(names, labels, strict=True)}
    return {}


# Pass 1: cluster a throwaway matrix to read flat labels from the dendrogram linkage.
_mat_cluster_probe = dega.clust.Matrix(transition_prob)
_mat_cluster_probe.cluster(force=True)
cluster_map = _station_clusters_from_matrix_dendrogram(_mat_cluster_probe, n_clusters=45)


In [ ]:
import pandas as pd

from celldega.clust.constants import _COLOR_PALETTE

_OKABE_ITO = (
    "#E69F00",
    "#56B4E9",
    "#009E73",
    "#F0E442",
    "#0072B2",
    "#D55E00",
    "#CC79A7",
)
STATION_PALETTE_HEX = list(_OKABE_ITO) + list(_COLOR_PALETTE)


def _bike_station_cluster_label(name: str, cmap: dict[str, int]) -> str:
    return str(int(cmap.get(str(name), 0)))


# Pass 2: new matrix with station_cluster in metadata from pass 1, then re-cluster for the widget.
meta_row = pd.DataFrame(
    {
        "station_cluster": [
            _bike_station_cluster_label(str(i), cluster_map) for i in transition_prob.index
        ]
    },
    index=transition_prob.index,
)
meta_col = pd.DataFrame(
    {
        "station_cluster": [
            _bike_station_cluster_label(str(i), cluster_map) for i in transition_prob.columns
        ]
    },
    index=transition_prob.columns,
)

mat = dega.clust.Matrix(
    transition_prob,
    meta_row=meta_row,
    meta_col=meta_col,
    row_attr=["station_cluster"],
    col_attr=["station_cluster"],
)
mat.cluster(force=True)
max_c = int(max(cluster_map.values(), default=0))
_station_cat_colors = {"0": "#b0b4be"}
for cid in range(1, max_c + 1):
    _station_cat_colors[str(cid)] = STATION_PALETTE_HEX[(cid - 1) % len(STATION_PALETTE_HEX)]
mat.set_global_cat_colors(_station_cat_colors)
mat.make_viz()

cgm = dega.viz.Clustergram(matrix=mat)


In [ ]:
import os

import anywidget
import pandas as pd
import traitlets

BIKE_DEBUG = True

# Nbconvert HTML: serve over http:// (e.g. `python -m http.server` in the download folder), not file:// -
# anywidget loaders and map tiles expect a normal origin. Rebuild the bundle: `npm run build:bike-map`.

# Bike map ESM bundle (anywidget `_esm`): always load from jsDelivr (GitHub `bikes` branch) so Jupyter matches nbconvert HTML.
# Override: set env BIKE_FLOW_MAP_ESM_URL to a full URL if you need another build (e.g. commit-pinned jsDelivr, or http://localhost:... during dev).

# jsDelivr URLs that use only the branch name (`@bikes`) can serve a stale bundle for days; pin the
# commit that contains `notebooks/bike_flow_map.widget.js`. After `npm run build:bike-map`, commit the
# bundle, run `git rev-parse HEAD`, and replace the SHA below (or set env BIKE_FLOW_MAP_ESM_URL).
BIKE_FLOW_MAP_ESM_DEFAULT = (
    "https://cdn.jsdelivr.net/gh/broadinstitute/celldega@70bdd44fb036de5b2cc56f8e4afedb4a981ae3a8/notebooks/bike_flow_map.widget.js"
)


from celldega.clust.constants import _COLOR_PALETTE

_OKABE_ITO = (
    "#E69F00",
    "#56B4E9",
    "#009E73",
    "#F0E442",
    "#0072B2",
    "#D55E00",
    "#CC79A7",
)
STATION_PALETTE_HEX = list(_OKABE_ITO) + list(_COLOR_PALETTE)

def _build_coord_df(stations: pd.DataFrame) -> pd.DataFrame:
    return (
        stations.groupby("station_name", as_index=False)
        .agg(
            lat=("lat", "mean"),
            lng=("lng", "mean"),
            station_id=(
                "station_id",
                lambda s: ", ".join(sorted({str(x) for x in s if pd.notna(x)})),
            ),
        )
    )


def _bike_flow_esm() -> str:
    """Return ESM URL: BIKE_FLOW_MAP_ESM_URL if set, else hosted jsDelivr default."""
    override = (os.environ.get("BIKE_FLOW_MAP_ESM_URL") or "").strip()
    return override or BIKE_FLOW_MAP_ESM_DEFAULT


class BikeFlowMapWidget(anywidget.AnyWidget):
    _esm = _bike_flow_esm()

    stations = traitlets.List(trait=traitlets.Dict(), default_value=[]).tag(sync=True)
    edge_index = traitlets.Dict(default_value={}).tag(sync=True)
    click_info = traitlets.Dict(default_value={}).tag(sync=True)
    selected_rows = traitlets.List(default_value=[]).tag(sync=True)
    selected_cols = traitlets.List(default_value=[]).tag(sync=True)
    width = traitlets.Int(560).tag(sync=True)
    height = traitlets.Int(800).tag(sync=True)
    debug = traitlets.Bool(False).tag(sync=True)
    palette_rgb = traitlets.List(default_value=[]).tag(sync=True)
    matrix_axis_slice = traitlets.Dict(default_value={}).tag(sync=True)
    matrix_slice_request_out = traitlets.Dict(default_value={}).tag(sync=True)
    cg_row_names = traitlets.List(default_value=[]).tag(sync=True)
    cg_col_names = traitlets.List(default_value=[]).tag(sync=True)


coord_df = _build_coord_df(stations)

cluster_map = _station_clusters_from_matrix_dendrogram(mat, n_clusters=200)
coord_df["cluster_id"] = (
    coord_df["station_name"].astype(str).map(cluster_map).fillna(0).astype(int)
)

flow = BikeFlowMapWidget(width=560, height=800, debug=BIKE_DEBUG)
flow.palette_rgb = [
    [int(h[i : i + 2], 16) for i in (1, 3, 5)] for h in STATION_PALETTE_HEX
]
flow.stations = [
    {
        "name": str(r.station_name),
        "lat": float(r.lat),
        "lng": float(r.lng),
        "station_id": str(r.station_id),
        "cluster_id": int(r.cluster_id),
    }
    for r in coord_df.itertuples()
]
flow.edge_index = {}


# Minimal bridge: forward clustergram interaction state to frontend.


# CGM → map: one-way dlink so the map can clear `click_info` / `matrix_axis_slice` without writing to CGM.
# Map → CGM: `matrix_slice_request_out` → `matrix_slice_request` (e.g. op `row_col` for row+col axis data).
from ipywidgets import HBox, Layout, dlink

dlink((cgm, "click_info"), (flow, "click_info"))
dlink((cgm, "selected_rows"), (flow, "selected_rows"))
dlink((cgm, "selected_cols"), (flow, "selected_cols"))
dlink((cgm, "matrix_slice_result"), (flow, "matrix_axis_slice"))
dlink((flow, "matrix_slice_request_out"), (cgm, "matrix_slice_request"))
dlink((cgm, "row_names"), (flow, "cg_row_names"))
dlink((cgm, "col_names"), (flow, "cg_col_names"))

flow.layout = Layout(width="560px", height="800px")
cgm.layout = Layout(width="720px", height="800px")
HBox([flow, cgm])


In [ ]:
1